In [1]:
# Regression from scratch (numpy + pandas only) - diabetes
import numpy as np, pandas as pd

df = pd.read_csv("../data/diabetes.csv")
X = df.drop(columns="disease_progression").to_numpy(float)
y = df["disease_progression"].to_numpy(float)

# split 80/20
np.random.seed(0)
i = np.random.permutation(len(X)); n = int(0.8 * len(X))
Xtr, Xte, ytr, yte = X[i[:n]], X[i[n:]], y[i[:n]], y[i[n:]]

# scale with TRAIN mean/std
mu, sd = Xtr.mean(0), Xtr.std(0)
Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd

In [2]:
# 1.1 models
bias = lambda A: np.c_[np.ones(len(A)), A]

def linreg(X, y, lam=0):                 # lam=0 -> Linear, lam>0 -> Ridge
    A = bias(X); I = np.eye(A.shape[1]); I[0, 0] = 0
    w = np.linalg.solve(A.T @ A + lam * I, A.T @ y)   # normal equation
    return lambda Xn: bias(Xn) @ w

def knn(X, y, k=10):
    def predict(Xn):
        d = ((Xn[:, None] - X[None]) ** 2).sum(2)     # distance to every train row
        return y[d.argsort(1)[:, :k]].mean(1)         # mean of k nearest
    return predict

models = {"Linear": linreg(Xtr, ytr), "Ridge": linreg(Xtr, ytr, lam=10), "KNN": knn(Xtr, ytr)}

In [3]:
# 1.1 benchmark / ranking (lower RMSE = better)
rmse = lambda y, p: np.sqrt(np.mean((y - p) ** 2))
r2   = lambda y, p: 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

res = pd.DataFrame({k: [rmse(yte, m(Xte)), r2(yte, m(Xte))] for k, m in models.items()},
                   index=["RMSE", "R2"]).T.sort_values("RMSE")
res

,RMSE,R2
Ridge,51.831610,0.606134
Linear,52.556467,0.595040
KNN,54.240156,0.568678


In [4]:
# 1.2 predict a new sample (order: age sex bmi bp s1 s2 s3 s4 s5 s6)
new = np.array([[50, 2, 28.0, 95, 200, 120, 50, 4.0, 4.8, 90]])
for k, m in models.items():
    print(k, m((new - mu) / sd)[0].round(1))

Linear 147.8
Ridge 151.7
KNN 144.8
